# ViT attention visualization

Load a trained ViT-B/16 checkpoint and plot the mean attention rollout
on a sample from Imagenette val. Follows Abnar & Zuidema 2020 (Attention
Rollout).


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from flax import nnx

from src.model.vit_flax import ViT, ViTConfig
from src.checkpoint.orbax_ckpt import build_checkpoint_manager

cfg = ViTConfig(num_classes=10)
model = ViT(cfg, rngs=nnx.Rngs(0))
graphdef, state = nnx.split(model)

# optional: restore a real checkpoint
# mgr = build_checkpoint_manager('outputs/ckpt-cpu-smoke')
# state = mgr.restore_latest({'state': state})['state']

In [ ]:
def collect_attn_weights(model, x):
    ws = []
    x = model.patch_embed(x)
    x = model.pos_embed(x)
    for blk in model.blocks:
        y = blk.ln1(x)
        _, w = blk.attn(y, y, need_weights=True, deterministic=True)
        ws.append(w)
        x = blk(x, deterministic=True)
    return ws

In [ ]:
def attention_rollout(weights, discard_ratio=0.0):
    result = None
    for w in weights:
        w_mean = w.mean(axis=1)  # avg over heads -> [B, N+1, N+1]
        w_mean = w_mean + jnp.eye(w_mean.shape[-1])
        w_mean = w_mean / w_mean.sum(axis=-1, keepdims=True)
        result = w_mean if result is None else jnp.matmul(w_mean, result)
    return result[:, 0, 1:]  # cls attention to patches


In [ ]:
# demo with a random image tensor
x = jnp.zeros((1, 224, 224, 3))
m = nnx.merge(graphdef, state)
ws = collect_attn_weights(m, x)
roll = attention_rollout(ws)
grid = np.asarray(roll).reshape(14, 14)
plt.imshow(grid, cmap='viridis')
plt.title('cls -> patch attention rollout')
plt.colorbar()
plt.show()